In [ ]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
#!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
#import os
#os.chdir('/content') # to avoid error if rerun
#os.chdir('./graph_representation_st457')

In [ ]:
# installation of Pytorch Geometric. This may take up to 30 minutes!!!

#!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
#!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
#!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# install DGL
#!pip install dgl
#!pip install numpy==1.26.4

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json

# load from other folders
from helper_functions import *
from model_classes import *

# Load data for GAT+RotatE

In [ ]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x, 
                                                                                                        batch_size =batch_size, 
                                                                                                        flatten_data = False, # Should be True for LSTM, False for GTC and GAT
                                                                                                        flatten_time_features=True # Should be True for GAT, False for LSTM and GTC
                                                                                                        )  


# Train GAT+RotatE model

In [ ]:
# GAT model in the current format cannot handle float values and need binary values. So I have set a threshold for correlation 
corr = adj_matrix[:, :, 2]
corr_bin = (corr.abs() > 0.5).float()
corr_bin.fill_diagonal_(1.0)

dict_adj_matrices = {'sector':    {'MSE':0, 'model':np.empty, 'matrix':adj_matrix[:,:,0].unsqueeze(-1), 'pred':np.empty},
                     'industry':  {'MSE':0, 'model':np.empty, 'matrix':adj_matrix[:,:,1].unsqueeze(-1), 'pred':np.empty},
                     'corre':     {'MSE':0, 'model':np.empty, 'matrix':corr_bin.unsqueeze(-1), 'pred':np.empty},
                     'everything':{'MSE':0, 'model':np.empty, 'matrix':torch.stack([adj_matrix[:, :, 0], adj_matrix[:, :, 1], corr_bin], dim=-1),               'pred':np.empty} }

for key in dict_adj_matrices.keys():
  print(f'Doing model: {key}')
  A_loop = dict_adj_matrices[key]['matrix']
  edge_index, edge_type = build_edges(A_loop)
  K_num_relations = A_loop.shape[-1] # This will be 1 for single relations, and 3 for 'everything'

  model_GAT_RotatE = GAT_RotatE(c_in=40, c_hidden=32, emb_dim=16, num_relations=K_num_relations,num_heads=4)
  optimizer = optim.Adam(model_GAT_RotatE.parameters(), lr=1e-3)

  num_epochs = 50 # My general observation is that it takes many more epochs to train this model
  train_losses = []
  val_losses = []

  for epoch in range(num_epochs):
      train_loss = train_one_epoch_GAT_RotatE(model_GAT_RotatE, train_loader, A_loop, edge_index, edge_type, optimizer)
      val_loss = evaluate_GAT_RotatE(model_GAT_RotatE, val_loader, A_loop)
      print(f"Epoch {epoch+1:03d} | train loss {train_loss:.6f} | val loss {val_loss:.6f}")

  dict_adj_matrices[key]['model'] = model_GAT_RotatE
  pred_test_GAT_RotatE = evaluate_GAT_RotatE(model_GAT_RotatE, test_loader, A_loop)

  dict_adj_matrices[key]['pred'] = pred_test_GAT_RotatE

  dict_adj_matrices[key]['MSE'] = np.mean((y_test-pred_test_GAT_RotatE.numpy())**2,axis=0)

  

In [ ]:
MSE_dict = {}
for key in dict_adj_matrices.keys():
    MSE_dict[key] = dict_adj_matrices[key]['MSE']